In [ ]:
# Instalación de dependencias geoespaciales en el entorno de Colab
!pip install pystac-client stackstac rioxarray geopandas rasterio pystac
!pip install odc-stac  #para el load()

In [ ]:
import os
import geopandas as gpd
from pystac_client import Client
import stackstac
import pandas as pd
import rioxarray
import numpy as np
from odc.stac import load
import xarray as xr
import glob
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from google.colab import drive;
# Esto fuerza al sistema a sincronizar los cambios con Drive
os.sync()
print("Sincronización forzada completada.")
from google.colab import drive
#drive.flush_and_unmount()
#print("✅ Drive desmontado correctamente.")
drive.mount('/content/drive')
from collections import Counter


Sincronización forzada completada.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# CONVIERTE DE TIF A PNG
def tif_to_png_batch(input_dir, output_dir, cmap_name='viridis', percentiles=(2, 98)):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"📁 Carpeta creada: {output_dir}")

    # Listar archivos .tif
    files = glob.glob(os.path.join(input_dir, "*.tif"))
    print(f"🚀 Procesando {len(files)} archivos...")

    for tif_path in files:
        filename = os.path.basename(tif_path).replace('.tif', '.png')
        save_path = os.path.join(output_dir, filename)
        with rasterio.open(tif_path) as src:
            data = src.read(1).astype(np.float32)# Leer la 1ra banda (suposición de índice escalar)
            nodata = src.nodata # Reemplazar valores NoData por NaN para el cálculo de estadísticas
            if nodata is not None:
                data[data == nodata] = np.nan

        # --- Optimización de Contraste ---
        vmin, vmax = np.nanpercentile(data, percentiles)# El uso de percentiles evita que píxeles erróneos saturen la imagen
        norm = Normalize(vmin=vmin, vmax=vmax, clip=True)# Normalización matemática al rango [0, 1] para Matplotlib

        # Generar la imagen con la paleta seleccionada
        plt.figure(figsize=(10, 10))
        plt.imshow(data, cmap=cmap_name, norm=norm)
        plt.axis('off')  # Eliminar ejes para presentación académica
        plt.savefig(save_path, bbox_inches='tight', pad_inches=0, dpi=300)# Guardar sin márgenes blancos
        plt.close() # Liberar memoria RAM del backend de pyplot
        print(f"✅ Convertido: {filename} [Paleta: {cmap_name}]")

In [ ]:
# 1. Definir el ROI exacto [lon_min, lat_min, lon_max, lat_max]
bbox_jujuy = [-64.89552688811091, -24.25718384771997, -64.84846538811773, -24.20934692727665]

# 2. Buscar en el catálogo de AWS
#sentinel-2-l2a (Reflectancia superficial (corregida atmosféricamente),
#sentinel-2-l1c — Reflectancia en tope de atmósfera
#sentinel-2-c1-l2a — Colección 1 L2A (más reciente en Earth Search)
print("🔍 Buscando imágenes Sentinel-2 L2A...")
catalog = Client.open("https://earth-search.aws.element84.com/v1")
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox_jujuy,
    datetime="2025-01-01/2025-12-31",
    query={"eo:cloud_cover": {"lt": 10}} # Máximo 10% de nubes
)
items = search.item_collection()
print(f"✅ Se encontraron {len(items)} escenas.")
# 3. Contar imágenes por mes
conteo = Counter(item.datetime.strftime("%Y-%m") for item in items)

# 4. Crear carpetas si no existen
base_dir = "/content/drive/MyDrive/aedesAegyptis2026/Sentinel_Jujuy_Data"
diario_dir = os.path.join(base_dir, "Stacks_Por_Fecha_test_2025_IBI")

if not os.path.exists(diario_dir):
    os.makedirs(diario_dir)
    print(f"📁 Carpeta creada: {diario_dir}")

# 5. Guardar reporte
reporte_path = os.path.join(diario_dir, "reporte_cant_imgs_disp2025.txt")

with open(reporte_path, "w", encoding="utf-8") as f:
    f.write("=" * 60 + "\n")
    f.write("   IMÁGENES DISPONIBLES POR MES — Sentinel-2 Jujuy\n")
    f.write("=" * 60 + "\n\n")

    for mes, cantidad in sorted(conteo.items()):
        print(f"  {mes}: {cantidad} imagen{'es' if cantidad > 1 else ''}\n")
        f.write(f"  {mes}: {cantidad} imagen{'es' if cantidad > 1 else ''}\n")

    f.write(f"\n  Total: {len(items)} escenas en el período.\n")
    f.write("\n" + "=" * 60 + "\n")

print(f"✅ Conteo guardado en: {reporte_path}")


In [ ]:
# 3. CARGA DEL CUBO DE DATOS (Data Cube)
try:
    ds =load(
        items,
        bands=["blue","green","red", "nir", "swir16", "swir22"],
        bbox=bbox_jujuy,
        crs="EPSG:4326",
        resolution=0.0001,
        groupby="solar_day",
        chunks={'time': 1, 'x': 512, 'y': 512} #bloques para no sobrecargar RAM
    )

# con esto tenes la pila de bandas, a partir de aqui computa los indices y por fecha vas iterando en ds
# acordate de normalizar cada banda por un factor de 10000
............

✅ ¡CONEXIÓN EXITOSA! Imágenes de Jujuy detectadas.
Dimensiones del área: FrozenMappingWarningOnValuesAccess({'latitude': 479, 'longitude': 472, 'time': 46})
